# Compliance QLoRA training
Select a GPU runtime. Stop the inference gateway before training to release VRAM. Upload the project as a ZIP, including inference/finetune/, then extract it under /content/sc-ai/platform. Training uses only cases you have reviewed and marked approved. Example cases are fictional and unapproved. This notebook does not upload adapters or modify production configuration.

In [ ]:
from pathlib import Path
ROOT = Path('/content/sc-ai/platform')
assert (ROOT / 'inference/finetune/compliance_tune.py').is_file(), 'Extract the project to /content/sc-ai/platform first.'

In [ ]:
%pip install -r /content/sc-ai/platform/inference/finetune/requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')

## Reviewed data
Upload your reviewed-cases.jsonl to /content. Do not mark examples approved without auditing their expected findings. Use independent company/process groups for train/eval; keep paraphrases together. Start with reviewed cases across all four grades plus ambiguous cases. No automatic training starts until you run the next cells.

In [ ]:
!python /content/sc-ai/platform/inference/finetune/compliance_tune.py prepare --input /content/reviewed-cases.jsonl --output /content/compliance-data

In [ ]:
!python /content/sc-ai/platform/inference/finetune/compliance_tune.py train --data /content/compliance-data --model Qwen/Qwen3.5-9B --output /content/compliance-adapter --max-length 2048

## Evaluate in separate fresh GPU sessions
Release the training model before each command (restart the session, reinstall dependencies, and restore data/adapter from your own storage). Compare the same held-out file and generation budget. Do not run both models alongside the gateway on a 15 GB GPU. Evaluate category agreement, NC-Major recall, invalid JSON and elapsed generation time before enabling an adapter.

In [ ]:
!python /content/sc-ai/platform/inference/finetune/compliance_tune.py evaluate --input /content/compliance-data/eval.jsonl --model Qwen/Qwen3.5-9B --output /content/base-report.json

In [ ]:
!python /content/sc-ai/platform/inference/finetune/compliance_tune.py evaluate --input /content/compliance-data/eval.jsonl --model Qwen/Qwen3.5-9B --adapter /content/compliance-adapter --output /content/adapter-report.json

## Enable only after evaluation
Back up /content/compliance-adapter before the runtime expires. In the inference notebook configuration set os.environ['AI_LORA_ADAPTER'] = '/content/compliance-adapter', keep AI_MODEL as the exact original base model, and restart/warm up the gateway. /health must show adapter_loaded=true. To roll back, clear AI_LORA_ADAPTER and restart. The backend still uses its normal HTTP provider. Validate actual application batches too; this offline evaluator uses single-control cases.